In [1]:
from datasets import load_dataset
# This downloads the data directly from Hugging Face
datasets = load_dataset("ag_news")
train_subset = datasets["train"]
test_subset = datasets["test"]

c:\Users\jan92\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# This function turns text into numbers because the model can only understand numbers.
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length")
# Apply this to our data
tokenized_train = train_subset.map(preprocess_function, batched=True)
tokenized_test = test_subset.map(preprocess_function, batched=True)

In [3]:
from transformers import AutoModelForSequenceClassification
#This is a pre-trained BERT model that is ready for sequence classification tasks (like categorizing news articles).
# We tell BERT to prepare for 4 specific categories (labels)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5601.15it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

In [4]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [5]:
from transformers import TrainingArguments

# Defining the rules of engagement for our model
training_args = TrainingArguments(
    output_dir="./bert_news_classifier",
    eval_strategy="epoch",        
    learning_rate=2e-5,                  # Standard safe learning rate for BERT
    per_device_train_batch_size=4,   
    per_device_eval_batch_size=4,
    fp16=True,                        # Use mixed precision for faster training (if supported)
    num_train_epochs=3,               
    weight_decay=0.01,         
    logging_steps=10,    
)

In [ ]:
from transformers import Trainer

# Assembling the pipeline
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Execute the training
print("Initiating fine-tuning sequence...")
trainer.train()

Initiating fine-tuning sequence...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.587475,0.403283,0.898000
2,0.286477,0.501862,0.888000
3,0.298902,0.491735,0.896000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]


TrainOutput(global_step=1500, training_loss=0.34354435714085896, metrics={'train_runtime': 481.4097, 'train_samples_per_second': 12.463, 'train_steps_per_second': 3.116, 'total_flos': 1578694680576000.0, 'train_loss': 0.34354435714085896, 'epoch': 3.0})

In [11]:
results = trainer.evaluate()
print(f"Model Accuracy: {results['eval_accuracy']:.2%}")
print(f"Model Loss: {results['eval_loss']:.2f}")

# To save the model and tokenizer for offline use
model.save_pretrained("./my_bert_news_model")
tokenizer.save_pretrained("./my_bert_news_model")
print("✅ Model saved! You can now use this offline anytime.")


Model Accuracy: 89.60%
Model Loss: 0.49


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.91it/s]

✅ Model saved! You can now use this offline anytime.
